# Anomaly Detection

We are building the first anomaly detection workflow for the AI4I 2020 dataset.

The goal is not to predict or classify the failure directly yet. The goal in this part is to detect abnormal machine behavior using anomaly detection (detect unknown faults).

The main concern that are tackled in this section is
- to distinguish what normal machine behavior from abnormal one?
- Which records look unusual compared with normal behavior?
- Do the unusual records overlap with actual machine failures?
- Can we explain why a record looks anomalous?

## Step 1: Import Libraries and Load Dataset

We are loading the dataset and required libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv('../../data/raw/ai4i2020.csv')
df.head()

,UID,Product ID,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Step 2: Select Safe Features

For anomaly detection, the model should only see machine condition features.

We do not use `Machine failure`, `TWF`, `HDF`, `PWF`, `OSF`, or `RNF` as input features. Those are labels used only for evaluation and interpretation.Those 5 sub-labels (`TWF`, `HDF`, `PWF`, `OSF`, and `RNF`) are derived from the same rules that generate `Machine failure`. If we leave them in as features when predicting Machine failure, we get near-perfect accuracy but it's actually data leakage, not a real model. So we drop these columns bofore training our data.

In [33]:
input_features = [
    'Type',
    'Air temperature',
    'Process temperature',
    'Rotational speed',
    'Torque',
    'Tool wear',
]

numeric_features = [
    'Air temperature',
    'Process temperature',
    'Rotational speed',
    'Torque',
    'Tool wear',
]

# First "responsible AI" checkpoint
# Data leakage detection is part of responsible modeling, not just accuracy engineering. 
# We are dropping these 5 columns before training on Machine failure.
X = df[input_features].copy()
y = df['Machine failure'].copy()
failure_modes = df[['TWF', 'HDF', 'PWF', 'OSF', 'RNF']].copy()

# Check target variable balance BEFORE preprocessing
print("Target variable balance (before split):")
print(df["Machine failure"].value_counts())
print(f"Failure rate: {df['Machine failure'].mean():.2%}")

print('X shape:', X.shape)
print("y shape:", y.shape)
X.head()

Target variable balance (before split):
Machine failure
0    9661
1     339
Name: count, dtype: int64
Failure rate: 3.39%
X shape: (10000, 6)
y shape: (10000,)


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


## Step 3: Train/Test Split

We split the data before encoding and scaling to avoid data leakage.

The labels are not used to train the anomaly detector, but we keep them so we can evaluate whether anomaly flags overlap with real failures.


In [34]:
# Train/test split before encoding and scaling
# Stratification keeps failure/no-failure proportions consistent in both sets

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size: {X_train_raw.shape[0]} samples")
print(f"Test set size: {X_test_raw.shape[0]} samples")
print(f"\nTraining set failure rate: {y_train.mean():.2%}")
print(f"Test set failure rate: {y_test.mean():.2%}")


Training set size: 8000 samples
Test set size: 2000 samples

Training set failure rate: 3.39%
Test set failure rate: 3.40%


## Step 4: Encode and Scale Features

`Type` is categorical, so we convert it into numeric columns by encoding.

The sensor columns have different units, so we standardize them. The scaler is fitted on the training set only, then applied to the test set.


In [35]:
# Encode categorical feature product Type
X_train_encoded = pd.get_dummies(X_train_raw, columns=['Type'], drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test_raw, columns=['Type'], drop_first=True, dtype=int)

# Making sure train and test have the same columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Using StandardScaler to scale numeric features of both test and train sets to have mean~0 and std=1
scaler = StandardScaler()

X_train_processed = X_train_encoded.copy()
X_test_processed = X_test_encoded.copy()

X_train_processed[numeric_features] = scaler.fit_transform(X_train_encoded[numeric_features])
X_test_processed[numeric_features] = scaler.transform(X_test_encoded[numeric_features])

print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)
X_train_processed.head()

Processed train shape: (8000, 7)
Processed test shape: (2000, 7)


,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Type_L,Type_M
4058,0.998914,0.604282,-0.460607,0.718305,-0.843997,0,1
1221,-1.505194,-1.153260,-0.775574,0.638456,0.382263,0,1
6895,0.498092,1.077466,-1.007654,0.558607,0.460870,0,1
9863,-0.553633,-0.139294,-0.709265,1.626586,-0.372359,1,0
8711,-1.455112,-1.018064,1.070019,-1.128202,-0.906882,1,0


## Step 5: Train on Normal Records

For this first anomaly detection experiment, we train the model using only normal records from the training set.

Intuition:

- normal records teach the model what typical machine behavior looks like
- records that do not fit this normal pattern may be flagged as anomalies

The failure labels are used here only to select normal training records for this educational experiment and to evaluate results afterward.


In [36]:
train_data_normal = y_train == 0
X_train_normal = X_train_processed.loc[train_data_normal]

print('All training records:', len(X_train_processed))
print('Normal training records used for anomaly detector:', len(X_train_normal))
print('Failure records excluded from anomaly detector training:', (~train_data_normal).sum())

All training records: 8000
Normal training records used for anomaly detector: 7729
Failure records excluded from anomaly detector training: 271


## Step 6: Fit the Anomaly Detection Model

The model returns:

- `1` for normal records
- `-1` for anomalous records

We convert this into:

- `0` for normal
- `1` for anomaly


In [37]:
anomaly_model = IsolationForest(
    contamination=0.05,
    random_state=42
)

anomaly_model.fit(X_train_normal)

,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


Now after model fit, we are going to predict the test data. So, we apply the model to the test set.

In [38]:
# prediction using the above fitted/trained model, now on the test set.
raw_predictions = anomaly_model.predict(X_test_processed)
anomaly_flag = (raw_predictions == -1).astype(int)

print('Normal test records predicted:', (anomaly_flag == 0).sum())
print('Anomalous test records predicted:', (anomaly_flag == 1).sum())
print('Anomaly rate:', f'{anomaly_flag.mean():.2%}')

Normal test records predicted: 1887
Anomalous test records predicted: 113
Anomaly rate: 5.65%


## Step 7: Compare Anomaly Flags With Actual Failures

Anomaly detection is not the same as failure classification.

A record can be unusual without being a failure, and a failure may not always look unusual. Still, we can check whether anomaly flags overlap with actual failures.


In [59]:
print('Confusion matrix: rows = actual failure, columns = predicted anomaly')
print(confusion_matrix(y_test, anomaly_flag))
print()

# print(y_test)
normal_test_data = y_test.copy(0)
failure_test_data = y_test.copy(1)
print(len(normal_test_data), len(failure_test_data))

print(anomaly_flag)

print(classification_report(y_test, anomaly_flag, target_names=['Normal', 'Failure/Anomaly'], zero_division=0))

Confusion matrix: rows = actual failure, columns = predicted anomaly
[[1835   97]
 [  52   16]]

2000 2000
[0 0 0 ... 0 0 0]
                 precision    recall  f1-score   support

         Normal       0.97      0.95      0.96      1932
Failure/Anomaly       0.14      0.24      0.18        68

       accuracy                           0.93      2000
      macro avg       0.56      0.59      0.57      2000
   weighted avg       0.94      0.93      0.93      2000



## Step 8: Inspect the Most Anomalous Records

The anomaly score tells us how unusual a record is. Lower scores indicate more unusual records.


In [66]:
anomaly_scores = anomaly_model.decision_function(X_test_processed)

results = X_test_raw.copy()
results['Machine failure'] = y_test.values
results['anomaly_flag'] = anomaly_flag
results['anomaly_score'] = anomaly_scores

most_anomalous = results.sort_values('anomaly_score').head(10)
most_anomalous

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,anomaly_flag,anomaly_score
989,M,296.2,307.0,2119,18.3,159,0,1,-0.095770
4296,L,301.8,310.2,2825,5.8,215,1,1,-0.082639
7997,M,301.0,312.2,2710,9.7,143,1,1,-0.079849
7086,L,300.6,310.4,2514,13.4,215,0,1,-0.078091
4899,L,303.6,312.3,2168,17.7,219,0,1,-0.078009
2080,L,299.3,309.2,2280,15.4,207,0,1,-0.069453
921,H,295.5,305.9,1593,37.2,197,0,1,-0.068339
8110,H,300.3,311.8,2141,18.5,204,0,1,-0.065517
4985,M,303.8,313.0,1789,23.5,215,0,1,-0.064318
380,L,297.5,308.3,2564,12.8,127,1,1,-0.063714


The anomaly detector appears to flag records with unusual operating conditions, especially records where rotational speed and torque were far from the normal machine behavior.

## Step 9: Simple Explanation of Anomalies

A simple explanation approach is to compare each anomalous record with the average normal training record.

For each record, we calculate how many standard deviations each numeric feature is from the normal training mean.

Large absolute values mean that feature is unusual.


In [67]:
normal_means = X_train_raw.loc[y_train == 0, numeric_features].mean()
normal_stds = X_train_raw.loc[y_train == 0, numeric_features].std()

def explain_record(row):
    z_scores = ((row[numeric_features] - normal_means) / normal_stds).abs()
    top_features = z_scores.sort_values(ascending=False).head(3)
    # print(', '.join([f'{feature}: {value:.2f} std away' for feature, value in top_features.items()]))
    return ', '.join([f'{feature}: {value:.2f} std away' for feature, value in top_features.items()])

explanation_table = most_anomalous.copy()
explanation_table['simple_explanation'] = explanation_table.apply(explain_record, axis=1)

explanation_table[[
    'Type',
    'Air temperature',
    'Process temperature',
    'Rotational speed',
    'Torque',
    'Tool wear',
    'Machine failure',
    'anomaly_flag',
    'anomaly_score',
    'simple_explanation'
]]

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,anomaly_flag,anomaly_score,simple_explanation
989,M,296.2,307.0,2119,18.3,159,0,1,-0.095770,"Rotational speed: 3.42 std away, Torque: 2.24 ..."
4296,L,301.8,310.2,2825,5.8,215,1,1,-0.082639,"Rotational speed: 7.60 std away, Torque: 3.56 ..."
7997,M,301.0,312.2,2710,9.7,143,1,1,-0.079849,"Rotational speed: 6.92 std away, Torque: 3.15 ..."
7086,L,300.6,310.4,2514,13.4,215,0,1,-0.078091,"Rotational speed: 5.76 std away, Torque: 2.76 ..."
4899,L,303.6,312.3,2168,17.7,219,0,1,-0.078009,"Rotational speed: 3.71 std away, Torque: 2.31 ..."
2080,L,299.3,309.2,2280,15.4,207,0,1,-0.069453,"Rotational speed: 4.38 std away, Torque: 2.55 ..."
921,H,295.5,305.9,1593,37.2,197,0,1,-0.068339,"Process temperature: 2.76 std away, Air temper..."
8110,H,300.3,311.8,2141,18.5,204,0,1,-0.065517,"Rotational speed: 3.55 std away, Torque: 2.22 ..."
4985,M,303.8,313.0,1789,23.5,215,0,1,-0.064318,"Process temperature: 2.03 std away, Air temper..."
380,L,297.5,308.3,2564,12.8,127,1,1,-0.063714,"Rotational speed: 6.06 std away, Torque: 2.82 ..."


## Step 10: Observations


After traning the model by fitting the train data and then testing the model with test data we are going to answer the folowing observations:

- How many records were flagged as anomalous?
- Did anomalies overlap with actual failures?
- Which features commonly explain anomalous records?
- Are there anomalies that are not failures?
- Are there failures that were not flagged as anomalies?
- What does this tell us about anomaly detection versus classification?

The anomaly detection model flagged a small portion of the test records as anomalous. The most anomalous records generally showed unusual sensor patterns, especially very high rotational speed, very low torque, high tool wear, or unusual temperature values. This suggests that the model is identifying records that differ from typical machine operating behavior.

Some of the top anomalous records also corresponded to actual machine failures. For example, records such as 4296, 7997, and 380 were flagged as anomalous and also had `Machine failure = 1`. However, several records were flagged as anomalous even though they did not have a failure label. This shows that anomaly detection is not the same as failure classification. An anomalous record means the machine behavior is unusual, but it does not always mean that a failure occurred.

This result supports the need for explainability. The anomaly flag alone only tells us that a record is unusual, but it does not explain why. By comparing anomalous records with normal machine behavior, we can identify which sensor features contributed most to the anomaly, such as rotational speed, torque, or tool wear. These explanations make the model output easier to understand and more useful for predictive maintenance analysis.


## Step 11: Milestone Summary Metrics

For the milestone, it is useful to report a few simple numbers:

- total test records
- number of records flagged as anomalous
- anomaly rate
- actual failures in the test set
- actual failures captured by anomaly flags

These numbers help explain what the anomaly detector is doing.

In [68]:
actual_failures = (y_test == 1).sum()
flagged_anomalies = anomaly_flag.sum()
failures_flagged_as_anomalies = ((y_test == 1) & (anomaly_flag == 1)).sum()
normal_records_flagged_as_anomalies = ((y_test == 0) & (anomaly_flag == 1)).sum()
failures_not_flagged = ((y_test == 1) & (anomaly_flag == 0)).sum()

summary_metrics = pd.DataFrame({
    'metric': [
        'total_test_records',
        'actual_failures_in_test_set',
        'records_flagged_as_anomalous',
        'anomaly_rate',
        'failures_flagged_as_anomalous',
        'normal_records_flagged_as_anomalous',
        'failures_not_flagged_as_anomalous'
    ],
    'value': [
        len(y_test),
        actual_failures,
        flagged_anomalies,
        f'{anomaly_flag.mean():.2%}',
        failures_flagged_as_anomalies,
        normal_records_flagged_as_anomalies,
        failures_not_flagged
    ]
})

summary_metrics

,metric,value
0,total_test_records,2000
1,actual_failures_in_test_set,68
2,records_flagged_as_anomalous,113
3,anomaly_rate,5.65%
4,failures_flagged_as_anomalous,16
5,normal_records_flagged_as_anomalous,97
6,failures_not_flagged_as_anomalous,52


## Step 12: Save Milestone Outputs

Saving results makes the notebook reproducible and gives us files to use in the milestone report.

We save:

- all test records with anomaly flags and scores
- the top anomalous records with simple explanations
- summary metrics

In [69]:
from pathlib import Path

output_dir = Path('../../outputs')
output_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(output_dir / 'anomaly_detection_test_results.csv', index=False)
explanation_table.to_csv(output_dir / 'top_anomaly_explanations.csv', index=False)
summary_metrics.to_csv(output_dir / 'anomaly_detection_summary_metrics.csv', index=False)

print('Saved files:')
print(output_dir / 'anomaly_detection_test_results.csv')
print(output_dir / 'top_anomaly_explanations.csv')
print(output_dir / 'anomaly_detection_summary_metrics.csv')

Saved files:
../../outputs/anomaly_detection_test_results.csv
../../outputs/top_anomaly_explanations.csv
../../outputs/anomaly_detection_summary_metrics.csv


## Step 13: Milestone 1 Conclusion Draft

Use this as a starting point for the milestone report:

The first milestone focused on detecting abnormal machine behavior using the AI4I 2020 Predictive Maintenance Dataset. The dataset was cleaned and preprocessed by selecting safe machine condition features, encoding the machine type, and scaling numeric sensor readings. An anomaly detection model was trained using normal machine records to learn typical operating behavior. The model then flagged unusual records in the test set.

The model flagged 113 of 2000 test records as anomalous, corresponding to an anomaly rate of 5.65%. Among the 68 actual machine failures in the test set, 16 were flagged as anomalous. This shows that anomaly detection can identify some failure-related abnormal behavior, but it does not capture all failures. Therefore, anomaly detection should be interpreted as a method for identifying unusual machine behavior rather than as a complete failure classifier. The results showed that some anomalous records overlapped with actual machine failures, while other anomalous records did not have a failure label. This confirms that anomaly detection is useful for identifying unusual operating behavior, but it should not be treated as a perfect failure classifier.

To support explainability, the most anomalous records were explained by comparing their sensor values with normal machine behavior. Features such as rotational speed, torque, tool wear, and temperature helped explain why certain records were flagged as anomalous. The explanation results show that the most anomalous records were mainly influenced by unusual rotational speed, torque, tool wear, and temperature values. For example, one failure case had rotational speed 7.60 standard deviations away from normal behavior and torque 3.56 standard deviations away. These feature-based explanations make the anomaly output more understandable.
